# Investigate photo-cluster conflicts

This notebook reads a completed repair work directory, shows clusters marked as conflicts, and reruns HDBSCAN or OPTICS with different parameters.

Trials are written to a separate directory. They never call `apply`, so no image is moved. Run `collect` once from the CLI before using this notebook.

In [ ]:
from pathlib import Path
import json
import shutil

%matplotlib inline
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image, ImageOps

from merge_group_metadata import capture_of, gps_of, index_metadata, read_metadata
from repair_library import aggregate, cluster

# Point this at the directory containing unique.json from the collect step.
WORK = Path("work")
TRIAL_WORK = Path("work-trial")
EXIFTOOL = "exiftool"

## Current results

Run this after `aggregate`. The table includes only groups with metadata conflicts. `group` is one-based and matches its entry in `clusters.json`.

In [ ]:
def load_results(work: Path):
    clusters = json.loads((work / "clusters.json").read_text())
    plans = json.loads((work / "plan.json").read_text())
    config_path = work / "clustering.json"
    config = json.loads(config_path.read_text()) if config_path.exists() else {}
    conflicts = [plan for plan in plans if plan.get("conflicts")]
    rows = [
        {
            "group": plan["group"],
            "members": len(plan["members"]),
            "max_hamming": plan.get("max_hamming", 0),
            "capture_time": plan.get("capture_time"),
            "conflicts": "; ".join(plan["conflicts"]),
            "pixel_source": plan["pixel_source"],
        }
        for plan in conflicts
    ]
    return clusters, plans, config, pd.DataFrame(rows)

clusters, plans, current_config, conflict_table = load_results(WORK)
print("Parameters:", current_config)
print(f"{len(clusters)} clusters; {len(conflict_table)} conflicting clusters")
display(conflict_table)

## Inspect one cluster

Choose a group number from the conflict table. The contact sheet labels the selected pixel source and each member's distance from the medoid.

In [ ]:
GROUP = int(conflict_table.iloc[0]["group"]) if not conflict_table.empty else 1
GROUP = conflict_table.iloc[10]["group"]
THUMBNAIL_SIZE = (260, 200)


def show_group(group_number: int, clusters=clusters, plans=plans):
    group = clusters[group_number - 1]
    plan = next(item for item in plans if item["group"] == group_number)
    members = group["members"]
    columns = min(4, len(members))
    rows = (len(members) + columns - 1) // columns
    figure, axes = plt.subplots(rows, columns, figsize=(4 * columns, 3.2 * rows), squeeze=False)
    for axis, member in zip(axes.flat, members):
        path = Path(member["path"])
        try:
            with Image.open(path) as image:
                preview = ImageOps.exif_transpose(image).convert("RGB")
                preview.thumbnail(THUMBNAIL_SIZE)
                axis.imshow(preview)
        except Exception as error:
            axis.text(0.5, 0.5, str(error), ha="center", va="center", wrap=True)
        selected = " [PIXELS]" if str(path) == plan["pixel_source"] else ""
        axis.set_title(f"d={member['hamming_to_medoid']}{selected}\n{path.name}", fontsize=9)
        axis.axis("off")
    for axis in axes.flat[len(members):]:
        axis.axis("off")
    figure.suptitle(
        f"Group {group_number}: max Hamming {group['max_hamming']}\n"
        + "; ".join(plan.get("conflicts", [])),
        fontsize=12,
    )
    figure.tight_layout()
    plt.show()
    display(pd.DataFrame({
        "path": plan["members"],
        "capture_source": [path == plan.get("capture_source") for path in plan["members"]],
        "gps_source": [path == plan.get("gps_source") for path in plan["members"]],
        "camera_source": [path == plan.get("camera_source") for path in plan["members"]],
    }))

show_group(GROUP)

In [ ]:
def show_member_metadata(group_number: int, clusters=clusters):
    paths = [member["path"] for member in clusters[group_number - 1]["members"]]
    metadata = index_metadata(read_metadata(EXIFTOOL, paths), paths)
    rows = []
    for path in paths:
        record = metadata.get(path, {})
        capture, capture_key = capture_of(record)
        rows.append({
            "path": path,
            "capture_time": capture,
            "capture_field": capture_key,
            "gps": gps_of(record),
            "camera": " ".join(
                str(record.get(key) or "").strip()
                for key in ("EXIF:Make", "EXIF:Model")
            ).strip(),
        })
    return pd.DataFrame(rows)

show_member_metadata(GROUP)

## Rerun clustering safely

Edit the parameters below, then run the next two cells. Distances are numbers of differing bits in the 256-bit perceptual hash.

- `min_cluster_size` and `min_samples` control density for both algorithms.
- `max_hamming` is a final explainable guard: members connected by more than this distance are split.
- HDBSCAN uses `hdbscan_epsilon`, `hdbscan_method`, and `allow_single_cluster`.
- OPTICS uses `optics_max_eps`, then either `xi` extraction (`optics_xi`) or DBSCAN extraction (`optics_eps`).

The cached perceptual hashes are copied to `TRIAL_WORK`, so images do not need to be decoded again.

In [ ]:
PARAMETERS = {
    "algorithm": "hdbscan",        # "hdbscan" or "optics"
    "min_cluster_size": 2,
    "min_samples": 2,
    "max_hamming": 32,
    # HDBSCAN parameters
    "hdbscan_epsilon": 0,           # hash bits
    "hdbscan_method": "eom",       # "eom" or "leaf"
    "allow_single_cluster": False,
    # OPTICS parameters
    "optics_max_eps": 64,           # hash bits
    "optics_cluster_method": "xi", # "xi" or "dbscan"
    "optics_xi": 0.05,
    "optics_eps": 32,               # hash bits; used by dbscan extraction
}
PARAMETERS

In [ ]:
TRIAL_WORK.mkdir(parents=True, exist_ok=True)
for filename in ("unique.json", "phash.json"):
    source = WORK / filename
    if not source.exists():
        raise FileNotFoundError(f"Run collection/clustering first; missing {source}")
    shutil.copy2(source, TRIAL_WORK / filename)

trial_clusters = cluster(TRIAL_WORK, **PARAMETERS)
trial_plans = aggregate(TRIAL_WORK, EXIFTOOL)
trial_clusters, trial_plans, trial_config, trial_conflicts = load_results(TRIAL_WORK)
print("Trial parameters:", trial_config)
print(f"{len(trial_clusters)} clusters; {len(trial_conflicts)} conflicting clusters")
display(trial_conflicts)

In [ ]:
comparison = pd.DataFrame([
    {
        "run": "current",
        "algorithm": current_config.get("algorithm", "unknown"),
        "clusters": len(clusters),
        "multi_image_clusters": sum(len(group["members"]) > 1 for group in clusters),
        "conflicts": len(conflict_table),
        "largest_cluster": max((len(group["members"]) for group in clusters), default=0),
    },
    {
        "run": "trial",
        "algorithm": trial_config["algorithm"],
        "clusters": len(trial_clusters),
        "multi_image_clusters": sum(len(group["members"]) > 1 for group in trial_clusters),
        "conflicts": len(trial_conflicts),
        "largest_cluster": max((len(group["members"]) for group in trial_clusters), default=0),
    },
])
display(comparison)

# Inspect any trial conflict without touching the original run.
TRIAL_GROUP = int(trial_conflicts.iloc[0]["group"]) if not trial_conflicts.empty else 1
show_group(TRIAL_GROUP, trial_clusters, trial_plans)
show_member_metadata(TRIAL_GROUP, trial_clusters)